# Forecast honesty audit: Colab runner

This notebook runs the full audit on **Chronos-Bolt** with real Indian data and produces the
interactive honesty report (`honesty_report.html`) used for the live demo.

**Before you start:** upload `honesty-audit.zip` using the folder icon on the left (Files pane).
A CPU runtime is enough for `chronos-bolt-small`; a T4 GPU only makes it faster.

In [ ]:
!unzip -oq /content/honesty-audit.zip -d /content
%cd /content/honesty-audit
!pip install -q chronos-forecasting

## Step 1: one forecast, to prove the model works

Downloads the last 60 days of hourly temperature for Navi Mumbai and asks Chronos-Bolt for the next 24 hours.
The shaded band is the model's own 80% range (its 0.1 and 0.9 quantiles).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from honesty_audit.data import fetch_open_meteo
from honesty_audit.models import ChronosBoltForecaster

s = fetch_open_meteo(19.03, 73.02, "2026-08-01", "2026-10-31", name="navi_mumbai")["navi_mumbai"]
model = ChronosBoltForecaster("amazon/chronos-bolt-small")
context, truth = s.values[-24*30-24:-24], s.values[-24:]
q = model.predict([context], horizon=24)[0]          # shape [24, 9]: quantiles 0.1 ... 0.9

x = np.arange(24)
plt.figure(figsize=(10, 4))
plt.plot(np.arange(-72, 0), context[-72:], color="grey", label="history")
plt.fill_between(x, q[:, 0], q[:, 8], alpha=.3, label="80% range")
plt.plot(x, q[:, 4], "--", label="median forecast")
plt.plot(x, truth, "k.", label="true value")
plt.legend(); plt.title("Chronos-Bolt, next 24 h, Navi Mumbai temperature"); plt.show()
inside = ((truth >= q[:, 0]) & (truth <= q[:, 8])).mean()
print(f"{inside:.0%} of the 24 true values fell inside the 80% range")

## Step 2 (optional): offline self-test

Checks the whole pipeline with synthetic data and a tiny stand-in network. Its numbers are **not** research results.

In [ ]:
!python -m pytest -q
!python -m honesty_audit run --model stand-in --data synthetic --context 168 --out results/selftest

## Step 3: the real audit

**Pre-register first.** Decide the pass/fail tolerance with Soham *before* running, write it in your research log,
and do not change it after seeing results. The default rule: an 80% range fails if we are 95% confident its coverage
is below 75% (`--tolerance 0.05`).

Data: hourly temperature for 10 Indian cities, June 2025 to October 2026 (Open-Meteo reanalysis).
The audit uses the most recent 40 days of each city: 20 days to fit the fix, the next 20 days to test.

In [ ]:
TOLERANCE = 0.05   # fixed in advance; record it in the research log
!python -m honesty_audit run --model chronos-bolt-small \
    --data open-meteo-cities:2025-06-01,2026-10-31 \
    --context 512 --horizon 24 --n-calib 20 --n-test 20 \
    --bits 16,8,6,4,3,2 --prune 0.3,0.5,0.7 --tolerance {TOLERANCE} --real-int8 \
    --out results/chronos_bolt_small_cities

## Step 4: read the results

In [ ]:
import json, pandas as pd
res = json.load(open("results/chronos_bolt_small_cities/results.json"))
rows = []
for v in res["variants"]:
    for m in ("raw", "split_cqr", "adaptive_cqr"):
        s = v["levels"]["0.8"][m]
        rows.append({"precision": v["name"], "range": m, "coverage": s["coverage"]["mean"],
                     "ci_lo": s["coverage"]["lo"], "ci_hi": s["coverage"]["hi"], "verdict": s["verdict"],
                     "width": s["scaled_width"], "interval_score": s["scaled_interval_score"]["mean"]})
df = pd.DataFrame(rows)
display(df.pivot(index="precision", columns="range", values="coverage").round(3))
pd.Series({k: f"safe to {v['safe_down_to']}, breaks at {v['breaks_at']}" for k, v in res["breaking_points"].items()})

In [ ]:
from google.colab import files
files.download("results/chronos_bolt_small_cities/honesty_report.html")   # open it on your laptop for the demo
files.download("results/chronos_bolt_small_cities/results.json")

## Step 5 (optional): second model, TimesFM-2.5

Install TimesFM by following the README at https://github.com/google-research/timesfm, then run the same command
with `--model timesfm-2.5`. Check that `forecast()` still returns the mean followed by the 0.1 to 0.9 quantiles;
`honesty_audit/models.py` assumes that layout.

## Step 6 (optional): your own data, e.g. Delhi electricity demand

Put a CSV in the Files pane with a time column and a value column, then:

```
!python -m honesty_audit run --model chronos-bolt-small --data csv:delhi_load.csv:load:timestamp --out results/delhi
```

Add a fourth part (`csv:file.csv:value:time:series_id`) if one file holds several series.